In [1]:
import glob
import os
import shutil

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from ultralytics import YOLO

import sys
sys.path.append("../../researches/objectdetectionmetrics/")
from metricsevaluator import MetricsEvaluator

In [2]:
def process_prediction(predictions):
    labels = predictions.boxes.cls.cpu().numpy()[predictions.boxes.cls.cpu().numpy() < 2]
    confidences = predictions.boxes.conf.cpu().numpy()[predictions.boxes.cls.cpu().numpy() < 2]
    boxes = predictions.boxes.xywhn.cpu().numpy()[predictions.boxes.cls.cpu().numpy() < 2]
    
    return os.path.basename(predictions.path).split(".")[0], labels, confidences, boxes

In [3]:
INPUT_WIDTH = 640
INPUT_HEIGHT = 640

def val_model(model_pth, imgs, labels, thres=0.35, iou_thres=0.45):
    model = YOLO(model_pth)
    predictions = [model.predict(img, conf=thres, iou=iou_thres, verbose=False)[0] for img in imgs]

    classes = ["chair", "table"]

    # init evaluator
    evaluator = MetricsEvaluator(
        is_relative_coordinates=True,
        is_xywh_coordinates=True,
        img_size=(INPUT_WIDTH, INPUT_HEIGHT),
        outpath="."
    )

    for label_pth in labels:
        img_name = label_pth.split("/")[-1].split(".")[0]

        with open(label_pth, "rt") as f:
            lines = f.readlines()
        
        for line in lines:
            c, x_c, y_c, w, h = [float(i) for i in line[:-1].split()][:5]
            if c > 1:
                continue
            if np.isnan(x_c):
                continue
            
            evaluator.add_gt_box(
                img_path=img_name,
                class_name=classes[int(c)],
                bbox=[x_c, y_c, w, h]
            )
        # break

    for img_predictions in tqdm(predictions):
        img_name, labels, confidences, boxes = process_prediction(img_predictions)
        for box, cl, conf in zip(boxes, labels, confidences):
            x_c = box[0] + box[2] / 2
            y_c = box[1] + box[3] / 2
            evaluator.add_det_box(
                img_path=img_name,
                class_name=classes[int(cl)],
                bbox=box,
                confidence=conf
            )

    results = evaluator.eval_conf_vals(0.5)

    for result in results:
        precision = result["total TP"] / (result["total TP"] + result["total FP"])
        recall = result["total TP"] / (result["total positives"])

        f_score = 2 * precision * recall / (precision + recall)
        print(f"{result['class']}: P = {precision:.03f}. R = {recall:.03f}. F1 = {f_score:.03f}. AP = {result['AP']:.03f}")


In [23]:
base_pth = "results"
model_name = "v2_batch6_merged_furniture_table_chair_model_s"
print(model_name)
print("=" * 50)


data = "datasets/v2_batch6/merged_furniture_table_chair"
print(data)

model_pth = os.path.join(base_pth, model_name, "weights/best.pt")
imgs = os.path.join(data, "images", "val")
imgs = [os.path.join(data, "images", "val", i) for i in os.listdir(os.path.join(data, "images", "val"))]

labels = os.listdir(os.path.join(data, "labels", "val"))
labels = [os.path.join(data, "labels", "val", label_pth) for label_pth in labels]

val_model(model_pth, imgs, labels)

data = "datasets/v3_2/table_chair/"
print(data)

model_pth = os.path.join(base_pth, model_name, "weights/best.pt")
imgs = os.path.join(data, "images", "val")
imgs = [os.path.join(data, "images", "val", i) for i in os.listdir(os.path.join(data, "images", "val"))]

labels = os.listdir(os.path.join(data, "labels", "val"))
labels = [os.path.join(data, "labels", "val", label_pth) for label_pth in labels]

val_model(model_pth, imgs, labels)


data = "datasets/v3_2/table_chair/"
print(data, "intersection")

intersection_ids = pd.read_csv("/home/klymy/data/ObjectDetectionV3_2/intersection_ids.csv")
intersection_ids["id"].values


imgs = os.path.join(data, "images", "val")
imgs = [os.path.join(data, "images", "val", i) for i in os.listdir(os.path.join(data, "images", "val")) 
        if i.split(".")[0] in intersection_ids["id"].values]

labels = os.listdir(os.path.join(data, "labels", "val"))
labels = [os.path.join(data, "labels", "val", label_pth) for label_pth in labels if 
          label_pth.split(".")[0] in intersection_ids["id"].values]

val_model(model_pth, imgs, labels)

data = "datasets/v2_batch6/merged_furniture_table_chair"
print(data, "reverse intersection")

intersection_ids = pd.read_csv("/home/klymy/data/ObjectDetectionV3_2/intersection_ids.csv")
intersection_ids["id"].values


imgs = os.path.join(data, "images", "val")
imgs = [os.path.join(data, "images", "val", i) for i in os.listdir(os.path.join(data, "images", "val")) 
        if i.split(".")[0] in intersection_ids["id"].values]

labels = os.listdir(os.path.join(data, "labels", "val"))
labels = [os.path.join(data, "labels", "val", label_pth) for label_pth in labels if 
          label_pth.split(".")[0] in intersection_ids["id"].values]

val_model(model_pth, imgs, labels)

v2_batch6_merged_furniture_table_chair_model_s
datasets/v2_batch6/merged_furniture_table_chair


100%|██████████| 292/292 [00:00<00:00, 854.25it/s]


chair: P = 0.871. R = 0.749. F1 = 0.806. AP = 0.726
table: P = 0.879. R = 0.710. F1 = 0.786. AP = 0.681
datasets/v3_2/table_chair/


100%|██████████| 459/459 [00:00<00:00, 834.67it/s]


chair: P = 0.895. R = 0.782. F1 = 0.834. AP = 0.763
table: P = 0.732. R = 0.606. F1 = 0.663. AP = 0.499
datasets/v3_2/table_chair/ intersection


100%|██████████| 227/227 [00:00<00:00, 853.39it/s]


chair: P = 0.888. R = 0.780. F1 = 0.830. AP = 0.761
table: P = 0.761. R = 0.653. F1 = 0.703. AP = 0.550
datasets/v2_batch6/merged_furniture_table_chair reverse intersection


100%|██████████| 2/2 [00:00<00:00, 515.08it/s]

chair: P = 0.826. R = 0.792. F1 = 0.809. AP = 0.728
table: P = 1.000. R = 0.727. F1 = 0.842. AP = 0.727


In [24]:
base_pth = "results"
model_name = "v3_2_table_chair"
print(model_name)
print("=" * 50)


data = "datasets/v2_batch6/merged_furniture_table_chair"
print(data)

model_pth = os.path.join(base_pth, model_name, "weights/best.pt")
imgs = os.path.join(data, "images", "val")
imgs = [os.path.join(data, "images", "val", i) for i in os.listdir(os.path.join(data, "images", "val"))]

labels = os.listdir(os.path.join(data, "labels", "val"))
labels = [os.path.join(data, "labels", "val", label_pth) for label_pth in labels]

val_model(model_pth, imgs, labels)

data = "datasets/v3_2/table_chair/"
print(data)

model_pth = os.path.join(base_pth, model_name, "weights/best.pt")
imgs = os.path.join(data, "images", "val")
imgs = [os.path.join(data, "images", "val", i) for i in os.listdir(os.path.join(data, "images", "val"))]

labels = os.listdir(os.path.join(data, "labels", "val"))
labels = [os.path.join(data, "labels", "val", label_pth) for label_pth in labels]

val_model(model_pth, imgs, labels)


data = "datasets/v3_2/table_chair/"
print(data, "intersection")

intersection_ids = pd.read_csv("/home/klymy/data/ObjectDetectionV3_2/intersection_ids.csv")
intersection_ids["id"].values


imgs = os.path.join(data, "images", "val")
imgs = [os.path.join(data, "images", "val", i) for i in os.listdir(os.path.join(data, "images", "val")) 
        if i.split(".")[0] in intersection_ids["id"].values]

labels = os.listdir(os.path.join(data, "labels", "val"))
labels = [os.path.join(data, "labels", "val", label_pth) for label_pth in labels if 
          label_pth.split(".")[0] in intersection_ids["id"].values]

val_model(model_pth, imgs, labels)

data = "datasets/v2_batch6/merged_furniture_table_chair"
print(data, "reverse intersection")

intersection_ids = pd.read_csv("/home/klymy/data/ObjectDetectionV3_2/intersection_ids.csv")
intersection_ids["id"].values


imgs = os.path.join(data, "images", "val")
imgs = [os.path.join(data, "images", "val", i) for i in os.listdir(os.path.join(data, "images", "val")) 
        if i.split(".")[0] in intersection_ids["id"].values]

labels = os.listdir(os.path.join(data, "labels", "val"))
labels = [os.path.join(data, "labels", "val", label_pth) for label_pth in labels if 
          label_pth.split(".")[0] in intersection_ids["id"].values]

val_model(model_pth, imgs, labels)

v3_2_table_chair
datasets/v2_batch6/merged_furniture_table_chair


100%|██████████| 292/292 [00:00<00:00, 876.13it/s]


chair: P = 0.855. R = 0.763. F1 = 0.806. AP = 0.737
table: P = 0.674. R = 0.560. F1 = 0.611. AP = 0.482
datasets/v3_2/table_chair/


100%|██████████| 459/459 [00:00<00:00, 837.98it/s]


chair: P = 0.876. R = 0.805. F1 = 0.839. AP = 0.785
table: P = 0.885. R = 0.772. F1 = 0.825. AP = 0.751
datasets/v3_2/table_chair/ intersection


100%|██████████| 227/227 [00:00<00:00, 849.61it/s]


chair: P = 0.877. R = 0.801. F1 = 0.837. AP = 0.781
table: P = 0.890. R = 0.787. F1 = 0.835. AP = 0.769
datasets/v2_batch6/merged_furniture_table_chair reverse intersection


100%|██████████| 2/2 [00:00<00:00, 654.03it/s]

chair: P = 0.875. R = 0.875. F1 = 0.875. AP = 0.844
table: P = 0.706. R = 0.545. F1 = 0.615. AP = 0.532


In [57]:
data = "datasets/v3_2/table_chair/"
print(data, "intersection")

intersection_ids = pd.read_csv("/home/klymy/data/ObjectDetectionV3_2/intersection_ids.csv")
intersection_ids["id"].values


labels_a = [i.split(".")[0] for i in os.listdir(os.path.join(data, "labels", "val"))]

data = "datasets/v2_batch6/merged_furniture_table_chair"

labels_b = [i.split(".")[0] for i in os.listdir(os.path.join(data, "labels", "val"))]

len(labels_a), len(labels_b)

datasets/v3_2/table_chair/ intersection


(459, 292)

In [58]:
intersection_ids["is_v3"] = intersection_ids["id_v3"].apply(lambda x: x in labels_b)
intersection_ids["is_v3_2"] = intersection_ids["id"].apply(lambda x: x in labels_a)

In [60]:
intersection_ids[intersection_ids["is_v3"] & intersection_ids["is_v3_2"]]

,id,name,id_v3,is_v3,is_v3_2
0,ccf4ffb0-b4bb-473f-ac70-51d1d051267f,PG1_CIR_H300_Meeting_Room_02_10_15_26_2016-9_2...,7d529d81-5a42-492c-baf4-aa647dbedd33,True,True
1,929d6377-1cfb-455f-82c4-2e488cbadae5,PG1_CIR_H300_Meeting_Room_02_10_15_26_2016-9_4...,6f827037-2d38-4be1-812b-30ccafe99476,True,True
2,46feacd4-fe13-43cd-8f2d-5c15eb346d3a,PG1_CIR_H300_Meeting_Room_02_11_21_28_2016-9_4...,438fd1e8-22de-462f-9659-b3532a9d05d5,True,True
3,042b03ca-75a7-4686-9632-1d31c8acffcf,PG1_CIR_H300_Meeting_Room_02_11_21_28_2016-9_5...,0aa2d334-cd55-440c-b7af-d188653107e3,True,True
4,c9c1ad45-f4ea-4b97-aaa4-15b56336c815,PG1_DEV_H250_MR_MindSpace_4Flr_side_01_1508326...,3d37f8db-3615-4efa-9bdb-e2346218dfef,True,True
...,...,...,...,...,...
224,543cec09-2b3d-4512-97c7-9ae1620ed5af,PG1_DL_H350_HD_NY_St170925_1499879317-18_318.jpg,1ffbbf3b-831b-4017-bc32-b2d7469ce546,True,True
225,abf57ec6-f39d-442d-9af1-1aecb22e19b4,PG1_H253_MR_PG_5min_4min_empty_1495116353-3_32...,dd8da859-9719-4050-beff-71aa3e66f761,True,True
226,8792c242-87d9-4b81-999e-ba9a9c8ca133,PG1_DL_H350_HD_NY_St170925_1499870831-18_1874.jpg,e756f702-b9a6-4fa4-b65f-e6c27fff986e,True,True
227,f57924e9-bab5-46c0-ad1f-2ec648171692,PG1_H250_LightTurnOn_After_1sec_PG_1498398198-...,81aa1a50-f09e-41bc-9255-dc8af97b776e,True,True


In [73]:
gt_c = 0

OUTPUT_PATH = f"intersection_{gt_c}"

if os.path.exists(OUTPUT_PATH):
    shutil.rmtree(OUTPUT_PATH)
os.makedirs(OUTPUT_PATH, exist_ok=True)


for idx, row in intersection_ids.iterrows():
    
    label_pth_v3 = os.path.join("datasets/v2_batch6/merged_furniture_table_chair/labels/val", f"{row['id_v3']}.txt")
    
    if os.path.exists(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.jpg")):
        img = cv2.imread(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.jpg"))
    elif os.path.exists(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.png")):
        img = cv2.imread(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.png"))
    elif os.path.exists(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.bmp")):
        img = cv2.imread(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.bmp"))
    else:
        print(row['id_v3'])
        continue
    
    img_h, img_w, _ = img.shape
    
    with open(label_pth_v3, "rt") as f:
        lines = f.readlines()
    
    for line in lines:
        c, x_c, y_c, w, h = [float(i) for i in line[:-1].split()][:5]
        if c != gt_c:
            continue
        if np.isnan(x_c):
            continue
        
        w = int(w * img_w)
        h  = int(h * img_h)
        x = int(x_c * img_w) - w // 2
        y = int(y_c * img_h) - h // 2
        cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 6)
        
    ####################
    label_pth_v3_2 = os.path.join("datasets/v3_2/table_chair/labels/val", f"{row['id']}.txt")

    with open(label_pth_v3_2, "rt") as f:
        lines = f.readlines()
    
    for line in lines:
        c, x_c, y_c, w, h = [float(i) for i in line[:-1].split()][:5]
        if c != gt_c:
            continue
        if np.isnan(x_c):
            continue
        
        w = int(w * img_w)
        h  = int(h * img_h)
        x = int(x_c * img_w) - w // 2
        y = int(y_c * img_h) - h // 2
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    # plt.imshow(img)
    cv2.imwrite(f"{OUTPUT_PATH}/{idx:>03}.jpg", img)
    # break

In [74]:
gt_c = 1

OUTPUT_PATH = f"intersection_{gt_c}"

if os.path.exists(OUTPUT_PATH):
    shutil.rmtree(OUTPUT_PATH)
os.makedirs(OUTPUT_PATH, exist_ok=True)


for idx, row in intersection_ids.iterrows():
    
    label_pth_v3 = os.path.join("datasets/v2_batch6/merged_furniture_table_chair/labels/val", f"{row['id_v3']}.txt")
    
    if os.path.exists(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.jpg")):
        img = cv2.imread(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.jpg"))
    elif os.path.exists(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.png")):
        img = cv2.imread(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.png"))
    elif os.path.exists(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.bmp")):
        img = cv2.imread(os.path.join("datasets/v2_batch6/merged_furniture_table_chair/images/val", f"{row['id_v3']}.bmp"))
    else:
        print(row['id_v3'])
        continue
    
    img_h, img_w, _ = img.shape
    
    with open(label_pth_v3, "rt") as f:
        lines = f.readlines()
    
    for line in lines:
        c, x_c, y_c, w, h = [float(i) for i in line[:-1].split()][:5]
        if c != gt_c:
            continue
        if np.isnan(x_c):
            continue
        
        w = int(w * img_w)
        h  = int(h * img_h)
        x = int(x_c * img_w) - w // 2
        y = int(y_c * img_h) - h // 2
        cv2.rectangle(img, (x, y), (x + w, y + h), (255, 0, 0), 6)
        
    ####################
    label_pth_v3_2 = os.path.join("datasets/v3_2/table_chair/labels/val", f"{row['id']}.txt")

    with open(label_pth_v3_2, "rt") as f:
        lines = f.readlines()
    
    for line in lines:
        c, x_c, y_c, w, h = [float(i) for i in line[:-1].split()][:5]
        if c != gt_c:
            continue
        if np.isnan(x_c):
            continue
        
        w = int(w * img_w)
        h  = int(h * img_h)
        x = int(x_c * img_w) - w // 2
        y = int(y_c * img_h) - h // 2
        cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    # plt.imshow(img)
    cv2.imwrite(f"{OUTPUT_PATH}/{idx:>03}.jpg", img)
    # break